In [1]:
"""
Generate & Store Vector Embeddings
===================================
Uses sentence-transformers to encode blog_posts.original_text into vector
embeddings, then stores them back into the same SQLite DB via sqlite-vector.

Multiple models are evaluated so embeddings can be compared later.
Column naming: <sanitized_model_name>_<dimension>
"""

import sqlite3
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────────────────
DB_PATH = (
    "/home/jain/Desktop/ws/public/generative_ai_workspace_2024_04_05/"
    "14_topic_classification_of_blog_posts_using_sqlite-vector/"
    "link_to_blog (20260703_0755).db"
)
VECTOR_EXT_PATH = "/home/jain/Desktop/cupboard/program_files/vector-linux-x86_64-1.0.0/vector.so"

TABLE_NAME = "blog_posts"
TEXT_COLUMN = "original_text"


In [2]:
# ── Connect to DB & Load sqlite-vector extension ──────────────────────
conn = sqlite3.connect(DB_PATH)
conn.enable_load_extension(True)
conn.load_extension(VECTOR_EXT_PATH)

# Verify extension is loaded
ver = conn.execute("SELECT vector_version()").fetchone()[0]
backend = conn.execute("SELECT vector_backend()").fetchone()[0]
print(f"sqlite-vector version: {ver}")
print(f"SIMD backend: {backend}")

# ── Fetch all rows ─────────────────────────────────────────────────────
rows = conn.execute(
    f"SELECT id, {TEXT_COLUMN} FROM {TABLE_NAME} WHERE {TEXT_COLUMN} IS NOT NULL"
).fetchall()

texts = [r[1] for r in rows]
row_ids = [r[0] for r in rows]
print(f"Fetched {len(texts)} rows with non-null '{TEXT_COLUMN}'")


sqlite-vector version: 1.0.0
SIMD backend: CPU
Fetched 79 rows with non-null 'original_text'


In [3]:
# ── Model registry ─────────────────────────────────────────────────────
# Each entry: (HuggingFace model ID, short sanitized name for the DB column)
# A diverse set for later comparison — different sizes, families, dims.

MODELS = [
    ("BAAI/bge-small-en-v1.5",             "bge_small_en_v1_5"),
    ("BAAI/bge-base-en-v1.5",              "bge_base_en_v1_5"),
    ("intfloat/e5-small-v2",               "e5_small_v2"),
    ("intfloat/e5-base-v2",                "e5_base_v2"),
    ("thenlper/gte-small",                 "gte_small"),
    ("thenlper/gte-base",                  "gte_base"),
    ("sentence-transformers/all-MiniLM-L6-v2",  "all_minilm_l6_v2"),
    ("sentence-transformers/all-MiniLM-L12-v2", "all_minilm_l12_v2"),
]

print(f"Will evaluate {len(MODELS)} models:")
for hf_name, short in MODELS:
    print(f"  • {hf_name}  →  column: {short}_<dim>")


Will evaluate 8 models:
  • BAAI/bge-small-en-v1.5  →  column: bge_small_en_v1_5_<dim>
  • BAAI/bge-base-en-v1.5  →  column: bge_base_en_v1_5_<dim>
  • intfloat/e5-small-v2  →  column: e5_small_v2_<dim>
  • intfloat/e5-base-v2  →  column: e5_base_v2_<dim>
  • thenlper/gte-small  →  column: gte_small_<dim>
  • thenlper/gte-base  →  column: gte_base_<dim>
  • sentence-transformers/all-MiniLM-L6-v2  →  column: all_minilm_l6_v2_<dim>
  • sentence-transformers/all-MiniLM-L12-v2  →  column: all_minilm_l12_v2_<dim>


In [4]:
# ── Generate & store embeddings for each model ─────────────────────────

failed_models = []

for hf_name, short in MODELS:
    print(f"\n{'='*70}")
    print(f"Model: {hf_name}")
    print(f"{'='*70}")

    # --- Load & encode (skip model entirely on any failure) ---
    try:
        print("  Loading …")
        model = SentenceTransformer(hf_name, trust_remote_code=True)

        dim = model.get_sentence_embedding_dimension()
        col_name = f"{short}_{dim}"
        print(f"  Dimension: {dim}  →  column: {col_name}")

        print(f"  Encoding {len(texts)} texts …")
        embeddings = model.encode(
            texts,
            batch_size=16,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )

        del model

    except Exception as e:
        print(f"  ✗ FAILED — {type(e).__name__}: {e}")
        print(f"  Skipping '{hf_name}' and continuing with next model.")
        failed_models.append(hf_name)
        continue

    # --- Store in DB ---
    print(f"  Storing into '{col_name}' …")

    try:
        conn.execute(f"ALTER TABLE {TABLE_NAME} ADD COLUMN {col_name} BLOB")
        print(f"    Added column.")
    except sqlite3.OperationalError:
        print(f"    Column already exists — overwriting.")

    for row_id, vec in tqdm(zip(row_ids, embeddings), total=len(row_ids), desc="  Updating"):
        conn.execute(
            f"UPDATE {TABLE_NAME} SET {col_name} = vector_as_f32(?) WHERE id = ?",
            (json.dumps(vec.tolist()), row_id),
        )

    conn.commit()

    # --- Register with sqlite-vector ---
    conn.execute(
        f"SELECT vector_init('{TABLE_NAME}', '{col_name}', "
        f"'type=FLOAT32,dimension={dim},distance=COSINE')"
    )
    print(f"  ✓ Done.\n")

# ── Summary ────────────────────────────────────────────────────────────
if failed_models:
    print(f"\n⚠ {len(failed_models)} model(s) failed to load:")
    for m in failed_models:
        print(f"    • {m}")
else:
    print("\n✓ All models loaded successfully.")



Model: BAAI/bge-small-en-v1.5
  Loading …


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

/tmp/ipykernel_14548/419687474.py:15: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

  Storing into 'all_minilm_l12_v2_384' …
    Added column.


  Updating: 100%|██████████| 79/79 [00:02<00:00, 34.92it/s]


  ✓ Done.


✓ All models loaded successfully.


In [5]:
# (Embeddings generated & stored above — proceed to verification ↓)


In [6]:
# ── Verify stored embeddings ───────────────────────────────────────────

print("\n" + "="*70)
print("VERIFICATION")
print("="*70)

# Get all columns, identify the embedding ones
cols = [c[1] for c in conn.execute(f"PRAGMA table_info({TABLE_NAME})").fetchall()]
NON_EMBEDDING_COLS = {
    "id", "url", "original_html", "original_text", "title",
    "new_shortened_text", "cover_image", "img_base64",
    "labels", "key_takeaways", "blogger_url", "ml_label"
}
embedding_cols = [c for c in cols if c not in NON_EMBEDDING_COLS]

print(f"Embedding columns found: {len(embedding_cols)}")
for col in embedding_cols:
    count = conn.execute(
        f"SELECT COUNT(*) FROM {TABLE_NAME} WHERE {col} IS NOT NULL"
    ).fetchone()[0]
    sample = conn.execute(
        f"SELECT length({col}) FROM {TABLE_NAME} WHERE {col} IS NOT NULL LIMIT 1"
    ).fetchone()
    blob_bytes = sample[0] if sample else 0
    inferred_dim = blob_bytes // 4  # float32 = 4 bytes/element
    print(f"  • {col}: {count}/{len(row_ids)} rows populated, "
          f"{blob_bytes} B blob → ~{inferred_dim} dims")

conn.close()
print("\n✓ All done. Database connection closed.")



VERIFICATION
Embedding columns found: 8
  • bge_small_en_v1_5_384: 79/79 rows populated, 1536 B blob → ~384 dims
  • bge_base_en_v1_5_768: 79/79 rows populated, 3072 B blob → ~768 dims
  • e5_small_v2_384: 79/79 rows populated, 1536 B blob → ~384 dims
  • e5_base_v2_768: 79/79 rows populated, 3072 B blob → ~768 dims
  • gte_small_384: 79/79 rows populated, 1536 B blob → ~384 dims
  • gte_base_768: 79/79 rows populated, 3072 B blob → ~768 dims
  • all_minilm_l6_v2_384: 79/79 rows populated, 1536 B blob → ~384 dims
  • all_minilm_l12_v2_384: 79/79 rows populated, 1536 B blob → ~384 dims

✓ All done. Database connection closed.
